In [22]:
# This Python 3 environment comes with many helpful analytics libraries installed
# It is defined by the kaggle/python Docker image: https://github.com/kaggle/docker-python
# For example, here's several helpful packages to load

import numpy as np # linear algebra
import pandas as pd # data processing, CSV file I/O (e.g. pd.read_csv)

# Input data files are available in the read-only "../input/" directory
# For example, running this (by clicking run or pressing Shift+Enter) will list all files under the input directory

import os
for dirname, _, filenames in os.walk('/kaggle/input'):
    for filename in filenames:
        print(os.path.join(dirname, filename))

# You can write up to 20GB to the current directory (/kaggle/working/) that gets preserved as output when you create a version using "Save & Run All" 
# You can also write temporary files to /kaggle/temp/, but they won't be saved outside of the current session

# Use the kagglehub client library to attach Kaggle resources like competitions, datasets, and models to your session
# Learn more about kagglehub: https://github.com/Kaggle/kagglehub/blob/main/README.md

import kagglehub
# kagglehub.dataset_download('<owner>/<dataset-slug>')

/kaggle/input/datasets/rondii/with-all-features/dataset_with_all_features.csv
/kaggle/input/datasets/rondii/with-all-features/dataset_with_all_features.pkl


In [23]:
df = pd.read_pickle('/kaggle/input/datasets/rondii/with-all-features/dataset_with_all_features.pkl')

In [24]:
df = df.loc[:, ~df.columns.duplicated(keep='first')].copy()

In [25]:
feature_cols = [c for c in df.columns if c != 'traffic_volume']
df[feature_cols] = df[feature_cols].shift(1)

In [26]:
df.replace([np.inf, -np.inf], np.nan, inplace=True)
df = df.dropna().reset_index(drop=True)

In [27]:
df[feature_cols] = df[feature_cols].astype('float64')

In [28]:
from sklearn.feature_selection import VarianceThreshold
from sklearn.metrics import normalized_mutual_info_score
from sklearn.preprocessing import KBinsDiscretizer
import warnings
warnings.filterwarnings('ignore')

In [29]:
X = df.drop(columns=['traffic_volume'])
y = df['traffic_volume']

In [30]:
vt = VarianceThreshold(threshold=1e-5)
X_vt = vt.fit_transform(X)
X = pd.DataFrame(X_vt, columns=X.columns[vt.get_support()], index=X.index)

In [31]:
def remove_correlated_features_mi(X, threshold=0.85, n_bins=20):
    discretizer = KBinsDiscretizer(n_bins=n_bins, encode='ordinal', strategy='quantile')
    X_binned = discretizer.fit_transform(X)
    X_binned = pd.DataFrame(X_binned, columns=X.columns, index=X.index)

    cols = X.columns.tolist()
    n = len(cols)
    to_drop = set()

    for i in range(n):
        if cols[i] in to_drop:
            continue
        vals_i = X_binned[cols[i]].values
        for j in range(i + 1, n):
            if cols[j] in to_drop:
                continue
            nmi = normalized_mutual_info_score(vals_i, X_binned[cols[j]].values)
            if nmi > threshold:
                to_drop.add(cols[i])
                break

    X_reduced = X.drop(columns=list(to_drop))
    print(f"Удалено признаков: {len(to_drop)} из {X.shape[1]}")
    return X_reduced

X = remove_correlated_features_mi(X, threshold=0.85, n_bins=20)

Удалено признаков: 52 из 253


In [32]:
from sklearn.feature_selection import RFE
from sklearn.model_selection import TimeSeriesSplit
import lightgbm as lgb

In [33]:
proxy_model = lgb.LGBMRegressor(
    n_estimators=30,
    max_depth=4,
    learning_rate=0.1,
    subsample=0.8,
    n_jobs=11,
    verbose=-1
)

In [34]:
rfe = RFE(
    estimator=proxy_model,
    n_features_to_select=15,
    step=5
)

rfe.fit(X, y)
selected_features_rfe = X.columns[rfe.get_support()].tolist()
print(selected_features_rfe)
print(len(selected_features_rfe))

['rolling_min_3', 'rolling_iqr_3', 'rolling_iqr_6', 'rolling_median_12', 'rolling_min_12', 'ema_0.1', 'ema_0.9', 'rsi_14', 'williams_r_14', 'momentum_1', 'atr_14', 'hour', 'hour_sin', 'hour_cos', 'dayofweek_sin']
15


In [35]:
df = X[selected_features_rfe].copy()
df['traffic_volume'] = y.values
df.to_pickle('/kaggle/working/dataset_selected_features.pkl')